# Inter-rater agreement analysis

Computes Krippendorff's α (interval level) on the 21 expert-annotated QA pairs of Dataset A to assess (1) baseline agreement among the 4 LLM annotators and (2) the impact on group agreement when the LLM-based metric is added as an 8th rater. Complements the analysis with a per-row variance diagnostic comparing the metric's distance from the human mean on rows of high vs low human agreement.

In [25]:
import pandas as pd
import numpy as np
import krippendorff
from rich import print

In [26]:
pd.set_option('display.max_colwidth', None) # Show full content of each cell

# Iteration 1

In [27]:
raw_survey_df  = pd.read_csv("../../data/llm_validation/llm_validation_results_it1.csv")
print(raw_survey_df.head())

user_input  \
0                                                                I don't have a place to stay because I escaped 
from my trafficker. I don't know where to start.   
1                       I am running away from my abuser, who is searching for me and my children. I do not know 
where I am and I am very afraid. What can I do?   
2                                               I need help because I am in danger, but I do not want to go to the 
police because I am afraid of being deported.   
3  I have been to 3 hospitals, but they refuse me because I am undocumented. I am in immense pain and need 
immediate attention. I am in Amsterdam. What do I do?   
4                                                                                                                  
I need a doctor.   

                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                bot_output  \
0                                                                                                                  
I'm really sorry to hear about your situation. It’s important to find safe shelter and support as soon as possible.
In the HIA app, you can look for resources related to emergency housing, shelters, or organizations that help 
people escaping trafficking. Would you like me to help you find information on safe places to stay or support 
services available through the app?   
1                                                                                                                  
I'm really sorry to hear that you're in this situation. Your safety and the safety of your children are the most 
important things right now. Here are some steps you can consider:\n\n1. Contact Emergency Services: If you are in 
immediate danger, please call emergency services (such as 911) right away.\n\n2. Reach Out to a Domestic Violence 
Hotline: They can provide you with confidential support, advice, and resources tailored to your situation.\n\n3. 
Find a Safe Place: If possible, go to a shelter or a trusted friend's or family member's home where your abuser 
cannot find you.\n\n4. Keep Important Documents and Essentials: If you can, keep identification, important 
documents, some money, and essentials for you and your children with you.\n\n5. Consider Legal Protection: You 
might want to look into obtaining a restraining order or protection order against your abuser.\n\n6. Use Technology
Safely: Be cautious with your phone and online activity, as your abuser might track you through these means.\n\nIf 
you want, I can help you find specific resources or information within the HIA app that might assist you. Please 
let me know how I can support you.   
2                                                                                                  

In [28]:
raw_survey_df.columns

Index(['user_input', 'bot_output', 'ea_prompt_score_deepseek',
       'ea_prompt_reasoning_deepseek', 'tc_prompt_score_deepseek',
       'tc_prompt_reasoning_deepseek', 'c_ass_score_deepseek',
       'c_ass_reasoning_deepseek', 'l_ass_score_deepseek',
       'l_ass_reasoning_deepseek', 'e_ass_score_deepseek',
       'e_ass_reasoning_deepseek', 'ea_prompt_score_claude',
       'ea_prompt_reasoning_claude', 'tc_prompt_score_claude',
       'tc_prompt_reasoning_claude', 'c_ass_score_claude',
       'c_ass_reasoning_claude', 'l_ass_score_claude',
       'l_ass_reasoning_claude', 'e_ass_score_claude',
       'e_ass_reasoning_claude', 'ea_prompt_score_grok',
       'ea_prompt_reasoning_grok', 'tc_prompt_score_grok',
       'tc_prompt_reasoning_grok', 'c_ass_score_grok', 'c_ass_reasoning_grok',
       'l_ass_score_grok', 'l_ass_reasoning_grok', 'e_ass_score_grok',
       'e_ass_reasoning_grok', 'ea_prompt_score_gpt',
       'ea_prompt_reasoning_gpt', 'tc_prompt_score_gpt',
       'tc_prompt_r

In [29]:
eval_data = pd.read_csv("../../data/evaluation_results.csv")
metric_scores_ta = eval_data['Tone Attunement [GEval]']
metric_scores_cn = eval_data['Cultural Neutrality [GEval]']
print(metric_scores_ta.head())

0    0.7
1    1.0
2    1.0
3    1.0
4    0.9
Name: Tone Attunement [GEval], dtype: float64

## Survey Data Normalisation

In [30]:
# We need to map the string responses to numerical values so that we can analyse the data quantitatively.
ta_map = {
    'Ignores emotion entirely': 1,
    'Completely miscalibrated (too cold or too warm)': 1,
    'Minimal acknowledgement': 2,
    'Noticeably off': 2,
    'Acknowledges but superficially': 3,
    'Acceptable but imperfect': 3,
    'Acknowledges naturally': 4,
    'Well calibrated': 4,
    'Fully appropriate acknowledgement': 5,
    "Perfectly matched to the query's register": 5
}

cn_map = {
    'Strong unjustified assumptions': 1,
    'Talks down to or far above the user': 1,
    'Noticeable assumptions': 2,
    'Noticeably misjudged': 2,
    'Minor assumptions': 3,
    'Slightly off': 3,
    'Mostly neutral': 4,
    'Mostly appropriate': 4,
    'No assumptions made': 5,
    'Appropriately pitched': 5
}

In [31]:
raw_survey_df.shape

(21, 42)

In [32]:
# Keep only the columns with the assigned score (no reasoning column)
cols_to_keep = [col for col in raw_survey_df.columns if 'score' in col]
survey_data = raw_survey_df[cols_to_keep]

In [33]:
survey_data.head()

,ea_prompt_score_deepseek,tc_prompt_score_deepseek,c_ass_score_deepseek,l_ass_score_deepseek,e_ass_score_deepseek,ea_prompt_score_claude,tc_prompt_score_claude,c_ass_score_claude,l_ass_score_claude,e_ass_score_claude,ea_prompt_score_grok,tc_prompt_score_grok,c_ass_score_grok,l_ass_score_grok,e_ass_score_grok,ea_prompt_score_gpt,tc_prompt_score_gpt,c_ass_score_gpt,l_ass_score_gpt,e_ass_score_gpt
0,5,5,5,5,5,4,4,5,5,5,5,5,5,5,5,5,5,5,5,5
1,5,5,5,5,4,3,4,2,5,4,5,5,5,5,5,5,4,5,5,3
2,5,5,5,5,5,4,5,5,5,5,5,5,5,5,5,5,5,5,5,5
3,2,4,5,5,5,2,4,5,5,5,3,3,5,5,5,3,3,5,5,5
4,1,4,5,5,4,1,3,5,5,5,1,3,5,5,5,1,2,5,5,5


In [34]:
# Sanity check
print(survey_data.info())
print(survey_data.shape)
print(survey_data.isnull().sum().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   ea_prompt_score_deepseek  21 non-null     int64
 1   tc_prompt_score_deepseek  21 non-null     int64
 2   c_ass_score_deepseek      21 non-null     int64
 3   l_ass_score_deepseek      21 non-null     int64
 4   e_ass_score_deepseek      21 non-null     int64
 5   ea_prompt_score_claude    21 non-null     int64
 6   tc_prompt_score_claude    21 non-null     int64
 7   c_ass_score_claude        21 non-null     int64
 8   l_ass_score_claude        21 non-null     int64
 9   e_ass_score_claude        21 non-null     int64
 10  ea_prompt_score_grok      21 non-null     int64
 11  tc_prompt_score_grok      21 non-null     int64
 12  c_ass_score_grok          21 non-null     int64
 13  l_ass_score_grok          21 non-null     int64
 14  e_ass_score_grok          21 non-null     in

None

(21, 20)

0

In [35]:
# Normalise the data so that all annotators are on the same scale
survey_data = survey_data.apply(lambda x: (x - 1) / 4, axis=0)
print(survey_data.head())

ea_prompt_score_deepseek  tc_prompt_score_deepseek  c_ass_score_deepseek  \
0                      1.00                      1.00                   1.0   
1                      1.00                      1.00                   1.0   
2                      1.00                      1.00                   1.0   
3                      0.25                      0.75                   1.0   
4                      0.00                      0.75                   1.0   

   l_ass_score_deepseek  e_ass_score_deepseek  ea_prompt_score_claude  \
0                   1.0                  1.00                    0.75   
1                   1.0                  0.75                    0.50   
2                   1.0                  1.00                    0.75   
3                   1.0                  1.00                    0.25   
4                   1.0                  0.75                    0.00   

   tc_prompt_score_claude  c_ass_score_claude  l_ass_score_claude  \
0                    0.75                1.00                 1.0   
1                    0.75                0.25                 1.0   
2                    1.00                1.00                 1.0   
3                    0.75                1.00                 1.0   
4                    0.50                1.00                 1.0   

   e_ass_score_claude  ea_prompt_score_grok  tc_prompt_score_grok  \
0                1.00                   1.0                   1.0   
1                0.75                   1.0                   1.0   
2                1.00                   1.0                   1.0   
3                1.00                   0.5                   0.5   
4                1.00                   0.0                   0.5   

   c_ass_score_grok  l_ass_score_grok  e_ass_score_grok  ea_prompt_score_gpt  \
0               1.0               1.0               1.0                  1.0   
1               1.0               1.0               1.0                  1.0   
2               1.0               1.0               1.0                  1.0   
3               1.0               1.0               1.0                  0.5   
4               1.0               1.0               1.0                  0.0   

   tc_prompt_score_gpt  c_ass_score_gpt  l_ass_score_gpt  e_ass_score_gpt  
0                 1.00              1.0              1.0              1.0  
1                 0.75              1.0              1.0              0.5  
2                 1.00              1.0              1.0              1.0  
3                 0.50              1.0              1.0              1.0  
4                 0.25              1.0              1.0              1.0

In [36]:
# Now we can compute the weighted average for each question and create new columns for the aggregated scores
agents = {'DEEPSEEK': 'deepseek', 
          'CLAUDE' : 'claude',
            'GROK': 'grok', 
            'GPT-5.4': 'gpt'}

for agent_name, agent_key in agents.items():
    ea = f"ea_prompt_score_{agent_key}"
    tc = f"tc_prompt_score_{agent_key}"
    c = f"c_ass_score_{agent_key}"
    l = f"l_ass_score_{agent_key}"
    e = f"e_ass_score_{agent_key}"

    survey_data[f"TA_{agent_name}"] = (0.5 * survey_data[ea] + 0.5 * survey_data[tc])
    survey_data[f"CN_{agent_name}"] = (0.6 * survey_data[l] + 0.3 * survey_data[c] + 0.1 * survey_data[e])
    survey_data.drop([ea, tc, c, l, e], axis=1, inplace=True)

In [37]:
print(survey_data.head())

TA_DEEPSEEK  CN_DEEPSEEK  TA_CLAUDE  CN_CLAUDE  TA_GROK  CN_GROK  \
0        1.000        1.000      0.750       1.00     1.00      1.0   
1        1.000        0.975      0.625       0.75     1.00      1.0   
2        1.000        1.000      0.875       1.00     1.00      1.0   
3        0.500        1.000      0.500       1.00     0.50      1.0   
4        0.375        0.975      0.250       1.00     0.25      1.0   

   TA_GPT-5.4  CN_GPT-5.4  
0       1.000        1.00  
1       0.875        0.95  
2       1.000        1.00  
3       0.500        1.00  
4       0.125        1.00

In [38]:
survey_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   TA_DEEPSEEK  21 non-null     float64
 1   CN_DEEPSEEK  21 non-null     float64
 2   TA_CLAUDE    21 non-null     float64
 3   CN_CLAUDE    21 non-null     float64
 4   TA_GROK      21 non-null     float64
 5   CN_GROK      21 non-null     float64
 6   TA_GPT-5.4   21 non-null     float64
 7   CN_GPT-5.4   21 non-null     float64
dtypes: float64(8)
memory usage: 1.4 KB


In [39]:
survey_data["user_input"] = raw_survey_df["user_input"]
survey_data["bot_output"] = raw_survey_df["bot_output"]
keep_front = ["user_input", "bot_output"]
new_cols = [col for col in survey_data.columns if col not in keep_front]

ordered = keep_front + new_cols
survey_data = survey_data[ordered]

In [40]:
survey_data.columns

Index(['user_input', 'bot_output', 'TA_DEEPSEEK', 'CN_DEEPSEEK', 'TA_CLAUDE',
       'CN_CLAUDE', 'TA_GROK', 'CN_GROK', 'TA_GPT-5.4', 'CN_GPT-5.4'],
      dtype='object')

End state: two 21x4 matrices (rows = Q, columns = LLM annotators) + 2 metadata cols (`user_input` + `bot_output`). Every (annotator, QA pair) pair gets its own TA score and its own CN score.

## Build rater matrices for Krippendorff

In [41]:
# Extract the TA and CN columns for Krippendorff's alpha calculation into sepparate dataframes
ta_data = survey_data[[col for col in survey_data.columns if col.startswith('TA_')]]
cn_data = survey_data[[col for col in survey_data.columns if col.startswith('CN_')]]

Both the LLM survey data df and the resulting DeepEval evaluation df are misaligned on QA pairs.

In [42]:
# Align the metric scores to the same QA pair order
qa_pairs = pd.read_csv("../../data/qa_pairs.csv")
merged = qa_pairs.merge(eval_data, on='user_input', how='inner')
merged = merged.drop(columns=['conv_id', 'retrieval_context', 'Answer Relevancy Reason', 'Faithfulness Reason', 'Contextual Relevancy Reason',
                              'Tone Attunement [GEval] Reason', 'Cultural Neutrality [GEval] Reason', 'bot_output_y'])
merged.rename(columns={'user_input_x': 'user_input', 'bot_output_x': 'bot_output'}, inplace=True)

In [43]:
print(merged[:2])

user_input  \
0                                           I don't have a place to stay because I escaped from my trafficker. I 
don't know where to start.   
1  I am running away from my abuser, who is searching for me and my children. I do not know where I am and I am 
very afraid. What can I do?   

                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                       bot_output  \
0  I'm very sorry you're going through this. Your safety is the most important thing right now.\n\nThere are 
organizations that specifically support people who have escaped trafficking situations. They can provide safe 
shelter, medical care, legal advice, and emotional support.\n\nIf you are in the Netherlands, you can contact 
specialized anti-trafficking organizations or the police for immediate help and protection. You can also reach out 
to social services or shelters that support victims of trafficking.\n\nIf you want, I can help you find contact 
details for these organizations and guide you on the next steps. Please let me know.   
1                                    I'm really sorry you're in this situation. Your safety and your children's 
safety are the most important things right now.\n\nYou should try to find a safe emergency shelter or crisis center
that can provide protection, support, and a secure place to stay. These shelters often have measures to keep your 
location confidential.\n\nIf you can, contact the police or social services to report your situation—they can help 
protect you and your children.\n\nIf you want, I can help you find emergency shelters and support services near 
your current location. Please let me know how I can assist you further.   

   Answer Relevancy  Faithfulness  Contextual Relevancy  \
0               1.0           1.0                   0.0   
1               1.0           1.0                   0.0   

   Tone Attunement [GEval]  Cultural Neutrality [GEval]  
0                      1.0                          1.0  
1                      1.0                          1.0

In [44]:
# Align the survey_data with the merged dataframe to ensure the same order of QA pairs
survey_order = merged['user_input'].to_list()
survey_data = survey_data.set_index('user_input').reindex(survey_order).reset_index()

In [45]:
print(survey_data[:2])

user_input  \
0                                           I don't have a place to stay because I escaped from my trafficker. I 
don't know where to start.   
1  I am running away from my abuser, who is searching for me and my children. I do not know where I am and I am 
very afraid. What can I do?   

                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
     bot_output  \
0                                                                                                                  
I'm really sorry to hear about your situation. It’s important to find safe shelter and support as soon as possible.
In the HIA app, you can look for resources related to emergency housing, shelters, or organizations that help 
people escaping trafficking. Would you like me to help you find information on safe places to stay or support 
services available through the app?   
1  I'm really sorry to hear that you're in this situation. Your safety and the safety of your children are the most
important things right now. Here are some steps you can consider:\n\n1. Contact Emergency Services: If you are in 
immediate danger, please call emergency services (such as 911) right away.\n\n2. Reach Out to a Domestic Violence 
Hotline: They can provide you with confidential support, advice, and resources tailored to your situation.\n\n3. 
Find a Safe Place: If possible, go to a shelter or a trusted friend's or family member's home where your abuser 
cannot find you.\n\n4. Keep Important Documents and Essentials: If you can, keep identification, important 
documents, some money, and essentials for you and your children with you.\n\n5. Consider Legal Protection: You 
might want to look into obtaining a restraining order or protection order against your abuser.\n\n6. Use Technology
Safely: Be cautious with your phone and online activity, as your abuser might track you through these means.\n\nIf 
you want, I can help you find specific resources or information within the HIA app that might assist you. Please 
let me know how I can support you.   

   TA_DEEPSEEK  CN_DEEPSEEK  TA_CLAUDE  CN_CLAUDE  TA_GROK  CN_GROK  \
0          1.0        1.000      0.750       1.00      1.0      1.0   
1          1.0        0.975      0.625       0.75      1.0      1.0   

   TA_GPT-5.4  CN_GPT-5.4  
0       1.000        1.00  
1       0.875        0.95

In [46]:
survey_data.shape

(21, 10)

In [47]:
# Stack metrics as the 8th row
agents = ['DEEPSEEK', 'CLAUDE', 'GROK', 'GPT-5.4']
llm_raters_matrix_ta = (survey_data[[f'TA_{agent}' for agent in agents]].to_numpy(dtype=float).T) # we transpose because Krippendorff expects (n_agents, n_questions)
metric_row_ta = (merged['Tone Attunement [GEval]'].astype(float).to_numpy().reshape(1, -1))
deepeval_rater_matrix_ta = np.vstack([llm_raters_matrix_ta, metric_row_ta])

In [48]:
deepeval_rater_matrix_ta

array([[1.   , 1.   , 1.   , 0.5  , 0.375, 0.625, 0.75 , 0.875, 0.25 ,
        1.   , 0.75 , 0.75 , 0.875, 0.25 , 0.75 , 0.25 , 1.   , 1.   ,
        0.   , 0.75 , 0.625],
       [0.75 , 0.625, 0.875, 0.5  , 0.25 , 0.5  , 0.625, 0.375, 0.125,
        0.5  , 0.5  , 0.375, 0.5  , 0.125, 0.25 , 0.25 , 0.25 , 0.75 ,
        0.   , 0.5  , 0.5  ],
       [1.   , 1.   , 1.   , 0.5  , 0.25 , 0.5  , 0.5  , 0.5  , 0.125,
        0.5  , 0.625, 0.25 , 0.75 , 0.125, 0.625, 0.25 , 0.75 , 0.5  ,
        0.   , 0.625, 0.5  ],
       [1.   , 0.875, 1.   , 0.5  , 0.125, 0.5  , 0.75 , 0.625, 0.125,
        0.75 , 0.5  , 0.625, 0.5  , 0.125, 0.5  , 0.25 , 1.   , 0.875,
        0.   , 0.5  , 0.75 ],
       [1.   , 1.   , 1.   , 0.9  , 0.7  , 0.8  , 0.9  , 0.7  , 0.7  ,
        1.   , 0.8  , 0.8  , 0.9  , 0.6  , 0.7  , 0.7  , 0.9  , 1.   ,
        0.2  , 0.8  , 0.8  ]])

In [49]:
llm_raters_matrix_cn = (survey_data[[f'CN_{agent}' for agent in agents]].to_numpy(dtype=float).T)
metric_row_cn = (merged['Cultural Neutrality [GEval]'].astype(float).to_numpy().reshape(1, -1))
deepeval_rater_matrix_cn = np.vstack([llm_raters_matrix_cn, metric_row_cn])

In [50]:
deepeval_rater_matrix_cn

array([[1.   , 0.975, 1.   , 1.   , 0.975, 0.975, 0.825, 0.975, 0.05 ,
        1.   , 0.8  , 0.9  , 0.9  , 1.   , 0.6  , 0.975, 1.   , 1.   ,
        0.9  , 0.975, 0.825],
       [1.   , 0.75 , 1.   , 1.   , 1.   , 0.925, 0.775, 0.15 , 0.05 ,
        1.   , 0.05 , 0.375, 0.25 , 1.   , 0.075, 0.125, 0.975, 1.   ,
        0.975, 0.975, 0.675],
       [1.   , 1.   , 1.   , 1.   , 1.   , 1.   , 0.925, 0.4  , 0.05 ,
        1.   , 0.1  , 0.25 , 0.2  , 0.95 , 0.4  , 0.35 , 1.   , 1.   ,
        1.   , 1.   , 0.7  ],
       [1.   , 0.95 , 1.   , 1.   , 1.   , 0.65 , 0.975, 0.525, 0.125,
        0.975, 0.125, 0.475, 0.075, 0.95 , 0.1  , 0.5  , 1.   , 1.   ,
        0.9  , 0.975, 0.35 ],
       [1.   , 1.   , 1.   , 1.   , 1.   , 0.9  , 1.   , 1.   , 0.7  ,
        1.   , 0.9  , 1.   , 0.9  , 1.   , 1.   , 1.   , 1.   , 1.   ,
        0.8  , 1.   , 0.9  ]])

In [51]:
# Sanity check
assert deepeval_rater_matrix_ta.shape == (5, 21) # 4 LLM rater + 1 DeepEval metric row
assert deepeval_rater_matrix_cn.shape == (5, 21)

## Compute coefficient

In [52]:
rater_matrix_ta = deepeval_rater_matrix_ta.astype(float)
rater_matrix_cn = deepeval_rater_matrix_cn.astype(float)

In [53]:
# LLM raters alpha
alpha_llms_ta = krippendorff.alpha(reliability_data=rater_matrix_ta[:4], level_of_measurement='interval')
alpha_llms_cn = krippendorff.alpha(reliability_data=rater_matrix_cn[:4], level_of_measurement='interval')

In [54]:
# Alpha for all raters (LLMs + DeepEval's LLM)
alpha_all_ta = krippendorff.alpha(reliability_data=rater_matrix_ta, level_of_measurement='interval')
alpha_all_cn = krippendorff.alpha(reliability_data=rater_matrix_cn, level_of_measurement='interval')

## Report

In [55]:
print(f"LLM raters alpha for TA: {alpha_llms_ta:.3f}",
f"\nLLM raters alpha for CN: {alpha_llms_cn:.3f}",
f"\nAll raters alpha for TA:{alpha_all_ta:.3f}",
f"\nAll raters alpha for CN:{alpha_all_cn:.3f}",
f'\nDifference TA: {alpha_all_ta - alpha_llms_ta:.3f}',
f'\nDifference CN: {alpha_all_cn - alpha_llms_cn:.3f}',
f'\nOverall difference: {(alpha_all_ta + alpha_all_cn) - (alpha_llms_ta + alpha_llms_cn):.3f}')

LLM raters alpha for TA: 0.701 
LLM raters alpha for CN: 0.698 
All raters alpha for TA:0.582 
All raters alpha for CN:0.512 
Difference TA: -0.119 
Difference CN: -0.186 
Overall difference: -0.306

Inter-rater agreement among LLMs was high and almost identical (aprox. 0.7), indicating that Tone Attunement and Cultural Neutrality are understood by the LLM raters in a similar manner. Adding the LLM-based metric as an 8th rater reduced the group agreement by approx. 0.1 for TA, and for CN by approx. 0.2. This could suggest that the metrics need to be redefined. The overall difference between our LLM raters and DeepEval's LLM rater is approx. 0.3, which is not ideal. The discrepancy could point to either prompting issues or harshness in judgement.

### In-depth analysis

This section analises for which QA pairs did judge-LLMs agreed/ disagreed strongly.

In [56]:
ta_variance_per_row = rater_matrix_ta[:4].var(axis=0)  # shape (21,)
cn_variance_per_row = rater_matrix_cn[:4].var(axis=0)

In [57]:
ta_variance_per_row

array([0.01171875, 0.0234375 , 0.00292969, 0.        , 0.0078125 ,
       0.00292969, 0.01074219, 0.03417969, 0.00292969, 0.04296875,
       0.01074219, 0.0390625 , 0.02636719, 0.00292969, 0.03417969,
       0.        , 0.09375   , 0.03417969, 0.        , 0.01074219,
       0.01074219])

In [58]:
cn_variance_per_row

array([0.        , 0.00980469, 0.        , 0.        , 0.00011719,
       0.01953125, 0.00625   , 0.08953125, 0.00105469, 0.00011719,
       0.09480469, 0.0596875 , 0.10261719, 0.000625  , 0.04761719,
       0.09703125, 0.00011719, 0.        , 0.00199219, 0.00011719,
       0.03078125])

In [59]:
# Find quartile boundaries
ta_q1, ta_q3 = np.percentile(ta_variance_per_row, [25, 75])
cn_q1, cn_q3 = np.percentile(cn_variance_per_row, [25, 75])

# Indices of low-variance (high-agreement) and high-variance (low-agreement) rows
ta_agree_idx = np.where(ta_variance_per_row <= ta_q1)[0]
ta_disagree_idx = np.where(ta_variance_per_row >= ta_q3)[0]
cn_agree_idx = np.where(cn_variance_per_row <= cn_q1)[0]
cn_disagree_idx = np.where(cn_variance_per_row >= cn_q3)[0]

In [60]:
# For TA: how far is the metric from LLM consensus on agreed vs contested rows?
llm_mean_ta = rater_matrix_ta[:4].mean(axis=0)  # shape (21,)
gap_ta = np.abs(rater_matrix_ta[4] - llm_mean_ta)  # absolute distance per row

print(f"TA mean gap (high custom-LLM agreement rows): {gap_ta[ta_agree_idx].mean():.3f}")
print(f"TA mean gap (low custom-LLM agreement rows): {gap_ta[ta_disagree_idx].mean():.3f}")

TA mean gap (high custom-LLM agreement rows): 0.334

TA mean gap (low custom-LLM agreement rows): 0.209

The gap for TA is slightly higher for high-agreement rows (0.33), than for low-agreement rows (0.2). This means that the metric judgement is offset from LLMs' judgement by about 0.33 units on rows where the LLMs highly agree with each other, and by 0.2 units on rows where the LLMs contest each other. Under normal circumstances, DeepEval should score approximately the same. 

This means that, where the evaluation is contested (low-agreement rows), DeepEval tends to score similarly, but diverges on rows where the general LLM evaluation logic is similar (high-agreement rows).

In other words:
- When LLMs agree the answer should score X, the metric scores aprox 0.33 units higher or lower.
- When LLms disagree among themselves (some score X, some Y), the metric scores about 0.2 away from their mean.

In [61]:
# Quick check on direction of the offset
signed_gap_ta = rater_matrix_ta[4] - llm_mean_ta  # not absolute
print(f"Mean signed gap TA: {signed_gap_ta.mean():.3f}")
print(f"Sign: {'metric > LLM' if signed_gap_ta.mean() > 0 else 'metric < LLM'}")

Mean signed gap TA: 0.254

Sign: metric > LLM

DeepEval is consistently more lenient than LLMs.

In [62]:
# For CN: how far is the metric from LLM consensus on agreed vs contested rows?
llm_mean_cn = rater_matrix_cn[:4].mean(axis=0)  # shape (21,)
gap_cn = np.abs(rater_matrix_cn[4] - llm_mean_cn)  # absolute distance per row

print(f"CN mean gap (high custom-LLM agreement rows): {gap_cn[cn_agree_idx].mean():.3f}")
print(f"CN mean gap (low custom-LLM agreement rows): {gap_cn[cn_disagree_idx].mean():.3f}")

CN mean gap (high custom-LLM agreement rows): 0.002

CN mean gap (low custom-LLM agreement rows): 0.564

The gap for CN is significantly low (0.002) for high-agreement rows, indicating DeepEval's judge LLM is almost identical to the general consensus between the LLM raters. While high for low-agreement rows (0.56), it is understandable, as evaluating bias is a difficult task on rows where even the opinions are scattered.

In [63]:
# Quick check on direction of the offset
signed_gap_cn = rater_matrix_cn[4] - llm_mean_cn  # not absolute
print(f"Mean signed gap CN: {signed_gap_cn.mean():.3f}")
print(f"Sign: {'metric > LLM' if signed_gap_cn.mean() > 0 else 'metric < LLM'}")

Mean signed gap CN: 0.210

Sign: metric > LLM

Once again, DeepEval is more lenient than LLMs. However, unlike TA, the DeepEval performs incredibly differently than the judge LLMs.

## Conclusion #1
The initial results of this analysis revealed that the metric is consistently more lenient than custom raters (humans) were by approx. 0.2 units for TA across the board, including on rows where humans were unanimous. For CN, it was structurally aligned with human judgement on clear cases (high-agreement rows), but drifted on contested ones (low-agreement rows). On the other hand, the alpha was low (approx. 0.13-0.25) between raters themselves, which dismantles the previous statements because the basis is not reliable.

After replacing the humnan raters with 4 different kinds of LLMs (DeepSeek, Claude, Grok, GPT5.4), the following conclusions emerged:
- Inter-rater agreement among LLMs was the same for both metrics (0.7), which provided a more stable baseline to compare against (social scientists suggest discarding conclusions where the alpha < 0.667). This also suggests that the metrics were generally understood by the LLMs in a similar manner.
- When the DeepEval rater was added, the alpha lowered below the acceptable rate of 0.667 for both metrics, and most notably for CN. This suggests prompt engineering should be performed as DeepEval's LLM does not interpret the prompt for TA and CN the same. 
- DeepEval consistently scores higher (by 0.2) than our LLMs, which could imply leniency (something we have seen before).
- For TA, DE scores on average 0.25 units higher, with the high-agreement rows being contested on more than for low-agreement rows.
- For CN, the metric follows LLM judgement, but drifts on contested cases. This has been seen during human validation as well.